In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Bidirectional, LSTM, Dense, Dropout,
    Lambda, Multiply, concatenate, Dot
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import mean_squared_error, mean_absolute_error, cohen_kappa_score
from sklearn.model_selection import train_test_split
import os
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# ── Environment Setup ─────────────────────────────────────────────────────────
DATASET_SLUG  = "siamese-data"
DATA_DIR      = f"/kaggle/input/{DATASET_SLUG}"
OUT_DIR       = "/kaggle/working"
USE_AUGMENTED = True             # True  → pakai final_*.npy + final_metadata.pkl
                                 # False → pakai data asli

gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow  : {tf.__version__}")
print(f"GPU tersedia: {len(gpus)}")
for g in gpus:
    print(" ", g)

prefix = 'final_' if USE_AUGMENTED else ''
meta_f = 'final_metadata.pkl' if USE_AUGMENTED else 'metadata.pkl'
for fname in [f'{prefix}questions_emb.npy', f'{prefix}answerkeys_emb.npy',
              f'{prefix}answers_emb.npy',    meta_f]:
    path   = os.path.join(DATA_DIR, fname)
    status = "OK" if os.path.exists(path) else "TIDAK DITEMUKAN"
    print(f"  {fname:<30} -> {status}")

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
# final_questions_emb & final_answerkeys_emb disimpan KOMPAK (1 per IDPSJ).
# Rekonstruksi array penuh menggunakan kolom psj_idx di metadata.

if USE_AUGMENTED:
    answers_emb    = np.load(os.path.join(DATA_DIR, 'final_answers_emb.npy'))
    uniq_q_emb     = np.load(os.path.join(DATA_DIR, 'final_questions_emb.npy'))
    uniq_ak_emb    = np.load(os.path.join(DATA_DIR, 'final_answerkeys_emb.npy'))
    metadata       = pd.read_pickle(os.path.join(DATA_DIR, 'final_metadata.pkl'))
else:
    answers_emb    = np.load(os.path.join(DATA_DIR, 'answers_emb.npy'))
    uniq_q_emb     = np.load(os.path.join(DATA_DIR, 'questions_emb.npy'))
    uniq_ak_emb    = np.load(os.path.join(DATA_DIR, 'answerkeys_emb.npy'))
    metadata       = pd.read_pickle(os.path.join(DATA_DIR, 'metadata.pkl'))

metadata = metadata.reset_index(drop=True)

# Jika data asli (belum kompak), psj_idx belum ada → buat sekarang
if 'psj_idx' not in metadata.columns:
    idpsj_sorted = sorted(metadata['IDPSJ'].unique())
    idpsj_to_idx = {psj: i for i, psj in enumerate(idpsj_sorted)}
    metadata['psj_idx'] = metadata['IDPSJ'].map(idpsj_to_idx)

# Rekonstruksi array penuh menggunakan psj_idx
questions_emb  = uniq_q_emb[metadata['psj_idx'].values]
answerkeys_emb = uniq_ak_emb[metadata['psj_idx'].values]

print("=== Hasil Load ===")
print(f"answers_emb    : {answers_emb.shape}    (per sampel)")
print(f"uniq_q_emb     : {uniq_q_emb.shape}   (kompak, per IDPSJ)")
print(f"questions_emb  : {questions_emb.shape}  (rekonstruksi)")
print(f"answerkeys_emb : {answerkeys_emb.shape} (rekonstruksi)")
print(f"\nMetadata       : {len(metadata)} rows")
print(f"Kolom metadata : {list(metadata.columns)}")
print(f"\nIDPSJ unik     : {sorted(metadata['IDPSJ'].unique())}")
print(f"\nDistribusi grade:")
print(metadata['grade'].value_counts().sort_index())

In [ ]:
# ── Hyperparameter ────────────────────────────────────────────────────────────
BILSTM_UNITS  = 128
DROPOUT       = 0.40
EPOCHS        = 150
BATCH_SIZE    = 16
PATIENCE      = 20
LR            = 2.68e-3
N_GRADES      = 10
N_THRESHOLDS  = N_GRADES - 1   # 9 sigmoid outputs

TRAIN_MODE = 'per_idpsj'   # 'per_idpsj' | 'pooled'

PER_PSJ_TEST_FRAC     = 0.20   # 20% test
PER_PSJ_VAL_FRAC      = 0.125  # 12.5% dari sisa 80% = 10% total
PER_PSJ_RANDOM_STATE  = 42

GLOBAL_TEST_FRAC     = 0.20
GLOBAL_VAL_FRAC      = 0.125  # 12.5% dari sisa 80% = 10% total
GLOBAL_RANDOM_STATE  = 42


# ── Ordinal Loss & Metric (identik dengan direct.ipynb v11) ───────────────────
def ordinal_loss(y_true, y_pred):
    """Sum of binary cross-entropies: apakah grade > k? (k=1..9)"""
    thresholds   = tf.cast(tf.range(1, 10), tf.float32)
    y_true_exp   = tf.expand_dims(tf.cast(y_true, tf.float32), -1)
    y_binary     = tf.cast(y_true_exp > thresholds, tf.float32)
    return tf.reduce_mean(
        tf.keras.losses.binary_crossentropy(y_binary, y_pred)
    )

def ordinal_mae(y_true, y_pred):
    """Grade prediction = jumlah threshold yang terlampaui + 1."""
    grade_pred = tf.reduce_sum(tf.cast(y_pred > 0.5, tf.float32), axis=-1) + 1.0
    return tf.reduce_mean(tf.abs(tf.cast(y_true, tf.float32) - grade_pred))


def attention_pool(seq_out, name_prefix):
    """Soft attention pooling atas output BiLSTM (return_sequences=True)."""
    score   = Dense(1, activation='tanh', use_bias=False,
                    name=f'{name_prefix}_attn_score')(seq_out)
    weights = Lambda(lambda x: tf.nn.softmax(x, axis=1),
                    name=f'{name_prefix}_attn_w')(score)
    pooled  = Lambda(lambda x: tf.reduce_sum(x[0] * x[1], axis=1),
                    name=f'{name_prefix}_attn_pool')([seq_out, weights])
    return pooled


def build_model(q_seq_len, ak_seq_len, a_seq_len, emb_dim=300,
                n_scalar=10,
                bilstm_units=BILSTM_UNITS, dropout=DROPOUT):
    """
    Siamese BiLSTM v11 — arsitektur identik dengan direct.ipynb (cross-prompt).
    Digunakan untuk evaluasi in-prompt sebagai pembanding kuantitatif.

    Merged: [ea(256), eak(256), eq(256), abs_diff(256), had_prod(256),
             cos_sim_ak_a(1), cos_sim_q_a(1), orisinalitas(1),
             scalar_dense(32)]  = 1315D
    Head: Dense(512) → Dropout → Dense(64) → Dense(9, sigmoid)  [ordinal]
    """
    shared_bilstm = Bidirectional(
        LSTM(bilstm_units, return_sequences=True), name='bilstm_shared'
    )

    inp_q      = Input(shape=(q_seq_len,  emb_dim), name='inp_q')
    inp_ak     = Input(shape=(ak_seq_len, emb_dim), name='inp_ak')
    inp_a      = Input(shape=(a_seq_len,  emb_dim), name='inp_a')
    inp_scalar = Input(shape=(n_scalar,),            name='inp_scalar')

    # ── BiLSTM + Attention Pooling ────────────────────────────────────────────
    eq_seq  = shared_bilstm(inp_q)
    eak_seq = shared_bilstm(inp_ak)
    ea_seq  = shared_bilstm(inp_a)

    eq  = attention_pool(eq_seq,  'q')
    eak = attention_pool(eak_seq, 'ak')
    ea  = attention_pool(ea_seq,  'a')

    # ── Kemiripan answerkey vs answer ─────────────────────────────────────────
    abs_diff     = Lambda(lambda x: tf.abs(x[0] - x[1]), name='abs_diff')([eak, ea])
    had_prod     = Multiply(name='had_prod')([eak, ea])
    cos_sim_ak_a = Dot(axes=1, normalize=True, name='cos_sim_ak_a')([eak, ea])

    # ── Relevansi jawaban terhadap pertanyaan ─────────────────────────────────
    cos_sim_q_a  = Dot(axes=1, normalize=True, name='cos_sim_q_a')([eq, ea])
    orisinalitas = Lambda(lambda x: 1.0 - x, name='orisinalitas')(cos_sim_q_a)

    # ── Scalar branch ─────────────────────────────────────────────────────────
    scalar_feat = Dense(32, activation='relu', name='scalar_dense')(inp_scalar)

    merged = concatenate(
        [ea, eak, eq, abs_diff, had_prod,
         cos_sim_ak_a, cos_sim_q_a, orisinalitas,
         scalar_feat],
        name='merged'
    )

    x   = Dense(512, activation='relu')(merged)
    x   = Dropout(dropout)(x)
    x   = Dense(64, activation='relu')(x)
    out = Dense(9, activation='sigmoid', name='ordinal_out')(x)

    model = Model(
        inputs=[inp_q, inp_ak, inp_a, inp_scalar],
        outputs=out,
        name='siamese_bilstm_v11_inprompt'
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
        loss=ordinal_loss,
        metrics=[ordinal_mae]
    )
    return model


# Verifikasi arsitektur (n_scalar=10: 5 normalized + 5 centroid)
_tmp = build_model(
    q_seq_len  = questions_emb.shape[1],
    ak_seq_len = answerkeys_emb.shape[1],
    a_seq_len  = answers_emb.shape[1],
    emb_dim    = answers_emb.shape[2],
    n_scalar   = 10
)
_tmp.summary()
del _tmp


In [ ]:
# ── Helper: scalar features (identik dengan direct.ipynb) ────────────────────
def compute_scalar_features_all(answers_emb, answerkeys_emb, verbose=True):
    n = answers_emb.shape[0]
    feats = np.zeros((n, 5), dtype=np.float32)
    if verbose:
        print(f"Precomputing scalar features untuk {n} sampel...")
    for i in range(n):
        if verbose and i % 500 == 0:
            print(f"  {i}/{n}")
        a  = answers_emb[i].astype(np.float64)
        ak = answerkeys_emb[i].astype(np.float64)

        ak_norm = ak / (np.linalg.norm(ak, axis=-1, keepdims=True) + 1e-8)
        a_norm  = a  / (np.linalg.norm(a,  axis=-1, keepdims=True) + 1e-8)

        ak_mask = np.abs(ak).sum(axis=-1) > 1e-6
        a_mask  = np.abs(a ).sum(axis=-1) > 1e-6
        ak_n    = ak_norm[ak_mask]
        a_n     = a_norm[a_mask]

        if ak_n.shape[0] == 0 or a_n.shape[0] == 0:
            continue

        sim = ak_n @ a_n.T
        rec = float(sim.max(axis=1).mean())
        pre = float(sim.max(axis=0).mean())
        f1  = 2.0 * rec * pre / (rec + pre + 1e-8)

        m_ak = ak_n.mean(axis=0)
        m_a  = a_n.mean(axis=0)
        cos  = float(m_ak @ m_a / (np.linalg.norm(m_ak) * np.linalg.norm(m_a) + 1e-8))
        lrat = float(a_mask.sum()) / max(float(ak_mask.sum()), 1.0)

        feats[i] = [rec, pre, f1, cos, lrat]

    if verbose:
        print(f"  Selesai. Shape: {feats.shape}")
    return feats


def make_scalar_input_train(feats, idpsj_ids):
    norm_part = np.zeros_like(feats)
    cent_part = np.zeros_like(feats)
    for idpsj in np.unique(idpsj_ids):
        m = idpsj_ids == idpsj
        mu = feats[m].mean(axis=0)
        sg = feats[m].std(axis=0) + 1e-8
        norm_part[m] = (feats[m] - mu) / sg
        cent_part[m] = mu
    norm_part = np.clip(norm_part, -3.0, 3.0)
    return np.hstack([norm_part, cent_part]).astype(np.float32)


def make_scalar_input_test(feats):
    mu = feats.mean(axis=0)
    sg = feats.std(axis=0) + 1e-8
    norm_part = np.clip((feats - mu) / sg, -3.0, 3.0)
    cent_part = np.tile(mu, (len(feats), 1))
    return np.hstack([norm_part, cent_part]).astype(np.float32)


# ── Precompute scalar features (1x untuk semua sampel) ───────────────────────
all_scalar_feats = compute_scalar_features_all(answers_emb, answerkeys_emb)
N_SCALAR = all_scalar_feats.shape[1] * 2   # 5 normalized + 5 centroid = 10

# ── Pelatihan ─────────────────────────────────────────────────────────────────
# per_idpsj : satu model tiap IDPSJ (split hanya dalam prompt itu).
#   Split  : 70% train | 10% val (early stopping) | 20% test
#   Train  = real (70%) + Mixup(syn_) dalam IDPSJ yang sama.
#   Val    = real (10%) — hanya untuk early stopping, bukan evaluasi.
#   Test   = real (20%) — evaluasi akhir.
#   CPI (cpi_) dikecualikan — jawaban pinjaman dari IDPSJ lain.

y_all = metadata['grade'].values.astype(np.float32)

if 'is_synthetic' in metadata.columns:
    is_real = ~metadata['is_synthetic'].values
else:
    is_real = ~metadata['IDJwb'].astype(str).str.startswith(('syn_', 'cpi_'))

fold_results = []

def get_split(arr, idx):
    return arr[idx].astype(np.float32)


def train_one_model(train_idx, val_idx, test_idx, tag, model_basename):
    """Latih satu model dengan early stopping; kembalikan dict hasil eval test."""
    y_train = y_all[train_idx]
    y_val   = y_all[val_idx]
    y_test  = y_all[test_idx]

    print(f"  Train: {len(y_train)}  |  Val: {len(y_val)}  |  Test: {len(y_test)}")

    X_q_tr   = get_split(questions_emb,  train_idx)
    X_ak_tr  = get_split(answerkeys_emb, train_idx)
    X_a_tr   = get_split(answers_emb,    train_idx)
    X_q_val  = get_split(questions_emb,  val_idx)
    X_ak_val = get_split(answerkeys_emb, val_idx)
    X_a_val  = get_split(answers_emb,    val_idx)
    X_q_te   = get_split(questions_emb,  test_idx)
    X_ak_te  = get_split(answerkeys_emb, test_idx)
    X_a_te   = get_split(answers_emb,    test_idx)

    # ── Scalar features ───────────────────────────────────────────────────────
    train_idpsj_ids = metadata['IDPSJ'].values[train_idx]
    scalar_tr  = make_scalar_input_train(all_scalar_feats[train_idx], train_idpsj_ids)
    scalar_val = make_scalar_input_test(all_scalar_feats[val_idx])
    scalar_te  = make_scalar_input_test(all_scalar_feats[test_idx])

    model = build_model(
        q_seq_len    = questions_emb.shape[1],
        ak_seq_len   = answerkeys_emb.shape[1],
        a_seq_len    = answers_emb.shape[1],
        emb_dim      = answers_emb.shape[2],
        n_scalar     = N_SCALAR,
        bilstm_units = BILSTM_UNITS,
        dropout      = DROPOUT,
    )

    grade_int          = y_train.astype(int)
    unique_g, counts_g = np.unique(grade_int, return_counts=True)
    freq_map           = dict(zip(unique_g, counts_g))
    n_kelas            = len(unique_g)
    raw_w              = np.array([len(y_train) / (n_kelas * freq_map[g]) for g in grade_int])
    sample_w           = raw_w / raw_w.mean()

    reduce_lr  = ReduceLROnPlateau(
        monitor='val_ordinal_mae', factor=0.5, patience=5,
        min_lr=1e-6, mode='min', verbose=0
    )
    early_stop = EarlyStopping(
        monitor='val_ordinal_mae', patience=PATIENCE,
        restore_best_weights=True, mode='min', verbose=1
    )

    model.fit(
        [X_q_tr, X_ak_tr, X_a_tr, scalar_tr], y_train,
        sample_weight=sample_w,
        validation_data=([X_q_val, X_ak_val, X_a_val, scalar_val], y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=[reduce_lr, early_stop], verbose=1
    )

    sigmoid_out = model.predict([X_q_te, X_ak_te, X_a_te, scalar_te], verbose=0)
    y_pred = np.clip(
        np.round(np.sum(sigmoid_out > 0.5, axis=-1) + 1), 1, 10
    ).astype(np.float32)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    qwk  = cohen_kappa_score(y_test.astype(int), y_pred.astype(int),
                              weights='quadratic')
    print(f"  MAE: {mae:.4f}  |  RMSE: {rmse:.4f}  |  QWK: {qwk:.4f}")

    model_path = os.path.join(OUT_DIR, model_basename)
    model.save(model_path)
    print(f"  Model saved -> {model_path}")
    tf.keras.backend.clear_session()

    return {
        'run'    : len(fold_results) + 1,
        'idpsj'  : tag,
        'mode'   : TRAIN_MODE,
        'n_train': len(y_train),
        'n_val'  : len(y_val),
        'n_test' : len(y_test),
        'mae'    : mae,
        'rmse'   : rmse,
        'qwk'    : qwk,
        'y_test' : y_test,
        'y_pred' : y_pred,
    }


# ── Pooled: satu model, semua prompt ─────────────────────────────────────────
if TRAIN_MODE == 'pooled':
    print("\n" + "=" * 60)
    print("Mode POOLED — satu model untuk semua IDPSJ")
    print("=" * 60)

    real_idx = metadata.index[is_real].values
    syn_idx  = metadata.index[~is_real].values

    tr_val, test_idx = train_test_split(
        real_idx, test_size=GLOBAL_TEST_FRAC,
        random_state=GLOBAL_RANDOM_STATE, shuffle=True,
    )
    train_real_idx, val_idx = train_test_split(
        tr_val, test_size=GLOBAL_VAL_FRAC,
        random_state=GLOBAL_RANDOM_STATE, shuffle=True,
    )
    train_idx = np.concatenate([train_real_idx, syn_idx])
    print(f"  Sintetis hanya di train: {len(syn_idx)}")

    fold_results.append(
        train_one_model(
            train_idx, val_idx, test_idx,
            tag='(semua IDPSJ)',
            model_basename='model_pooled.keras',
        )
    )
    print("\nSelesai: 1 model (pooled).")

# ── Per IDPSJ: in-prompt evaluation ──────────────────────────────────────────
elif TRAIN_MODE == 'per_idpsj':
    idpsj_list = sorted(metadata['IDPSJ'].unique())
    n_parts    = len(idpsj_list)

    is_mixup = metadata['IDJwb'].astype(str).str.startswith('syn_')

    for i, psj_id in enumerate(idpsj_list):
        print(f"\n{'='*60}")
        print(f"IDPSJ {i+1:02d}/{n_parts}  |  prompt = {psj_id}")

        real_idx = metadata.index[
            (metadata['IDPSJ'] == psj_id) & is_real
        ].values
        syn_idx = metadata.index[
            (metadata['IDPSJ'] == psj_id) & is_mixup
        ].values

        if len(real_idx) < 5:
            print(f"  Lewati: kurang dari 5 sampel asli (n={len(real_idx)})")
            continue

        rs = PER_PSJ_RANDOM_STATE + i
        tr_val, test_idx = train_test_split(
            real_idx, test_size=PER_PSJ_TEST_FRAC,
            random_state=rs, shuffle=True,
        )
        if len(tr_val) < 2:
            print("  Lewati: terlalu sedikit sampel setelah split test")
            continue

        train_real_idx, val_idx = train_test_split(
            tr_val, test_size=PER_PSJ_VAL_FRAC,
            random_state=rs, shuffle=True,
        )
        train_idx = np.concatenate([train_real_idx, syn_idx])
        print(f"  (Mixup di train: {len(syn_idx)})")

        safe_id = str(psj_id).replace(os.sep, '_').replace('/', '_')
        fold_results.append(
            train_one_model(
                train_idx, val_idx, test_idx,
                tag=psj_id,
                model_basename=f'model_idpsj_{i+1:02d}_{safe_id}.keras',
            )
        )

    print(f"\n\nSelesai: {len(fold_results)} model (per IDPSJ yang memenuhi syarat).")

else:
    raise ValueError("TRAIN_MODE harus 'pooled' atau 'per_idpsj'")


In [ ]:
# ── Evaluasi Akhir ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

summary = pd.DataFrame([{
    'idpsj'  : r['idpsj'],
    'n_train': r['n_train'],
    'n_test' : r['n_test'],
    'MAE'    : round(r['mae'],  4),
    'RMSE'   : round(r['rmse'], 4),
    'QWK'    : round(r['qwk'],  4),
} for r in fold_results])

print("=" * 65)
print("Hasil per IDPSJ — Siamese BiLSTM v11 In-Prompt")
print("=" * 65)
print(summary.to_string(index=False))
print(f"\nMAE  : {summary['MAE'].mean():.4f}  ±  {summary['MAE'].std():.4f}")
print(f"RMSE : {summary['RMSE'].mean():.4f}  ±  {summary['RMSE'].std():.4f}")
print(f"QWK  : {summary['QWK'].mean():.4f}  ±  {summary['QWK'].std():.4f}")

summary.to_csv(os.path.join(OUT_DIR, 'inprompt_results_v11.csv'), index=False)

y_all_true = np.concatenate([r['y_test'] for r in fold_results])
y_all_pred = np.concatenate([r['y_pred'] for r in fold_results])
x          = np.arange(len(summary))
idpsj_labels = summary['idpsj'].astype(str)

# ── Gambar 1: MAE per IDPSJ ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x, summary['MAE'], color='steelblue', alpha=0.85,
       edgecolor='black', linewidth=0.5)
ax.axhline(summary['MAE'].mean(), color='red', linestyle='--', linewidth=1.5,
           label=f"Mean MAE = {summary['MAE'].mean():.4f}")
ax.set_xticks(x)
ax.set_xticklabels(idpsj_labels)
ax.set_xlabel('IDPSJ')
ax.set_ylabel('MAE')
ax.set_title('MAE per IDPSJ — Siamese BiLSTM v11 In-Prompt')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'inprompt_plot1_mae.png'), dpi=150)
plt.show()

# ── Gambar 2: Scatter Plot Prediksi vs Aktual ────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_all_true, y_all_pred, alpha=0.4, edgecolors='k', linewidths=0.3)
ax.plot([1, 10], [1, 10], 'r--', label='Ideal')
ax.set_xlabel('Grade Aktual')
ax.set_ylabel('Grade Prediksi')
ax.set_title(f'Prediksi vs Aktual — Siamese BiLSTM v11 In-Prompt\n'
             f'MAE={summary["MAE"].mean():.4f}  '
             f'RMSE={summary["RMSE"].mean():.4f}  '
             f'QWK={summary["QWK"].mean():.4f}')
ax.set_xticks(range(1, 11))
ax.set_yticks(range(1, 11))
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'inprompt_plot2_scatter.png'), dpi=150)
plt.show()

# ── Gambar 3: RMSE per IDPSJ ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x, summary['RMSE'], color='mediumseagreen', alpha=0.85,
       edgecolor='black', linewidth=0.5)
ax.axhline(summary['RMSE'].mean(), color='red', linestyle='--', linewidth=1.5,
           label=f"Mean RMSE = {summary['RMSE'].mean():.4f}")
ax.set_xticks(x)
ax.set_xticklabels(idpsj_labels)
ax.set_xlabel('IDPSJ')
ax.set_ylabel('RMSE')
ax.set_title('RMSE per IDPSJ — Siamese BiLSTM v11 In-Prompt')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'inprompt_plot3_rmse.png'), dpi=150)
plt.show()

# ── Gambar 4: QWK per IDPSJ ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x, summary['QWK'], color='mediumpurple', alpha=0.85,
       edgecolor='black', linewidth=0.5)
ax.axhline(summary['QWK'].mean(), color='darkviolet', linestyle='--', linewidth=1.5,
           label=f"Mean QWK = {summary['QWK'].mean():.4f}")
ax.axhline(0.6, color='gray', linestyle=':', linewidth=1.2,
           label='Threshold 0.6 (substantial)')
ax.set_xticks(x)
ax.set_xticklabels(idpsj_labels)
ax.set_xlabel('IDPSJ')
ax.set_ylabel('QWK')
ax.set_ylim(0, 1)
ax.set_title('Quadratic Weighted Kappa per IDPSJ — Siamese BiLSTM v11 In-Prompt')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'inprompt_plot4_qwk.png'), dpi=150)
plt.show()

print("\nGambar disimpan:")
for name in ['inprompt_plot1_mae.png', 'inprompt_plot2_scatter.png',
             'inprompt_plot3_rmse.png', 'inprompt_plot4_qwk.png']:
    print(f"  {os.path.join(OUT_DIR, name)}")
